<a href="https://colab.research.google.com/github/ChandrikaImmadi/GenAI-Tasks/blob/main/Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu openai

In [ ]:
import os
import openai
import faiss
import numpy as np
from pypdf import PdfReader

In [ ]:
os.environ["OPENAI_API_KEY"] = "API_KEY"
openai.api_key = os.environ["OPENAI_API_KEY"]

In [ ]:

from google.colab import files

uploaded = files.upload()


for filename in uploaded.keys():
    print(f"Uploaded file: {filename}")


pdf_path = list(uploaded.keys())[0]
print("PDF path:", pdf_path)


In [ ]:
reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(text)

In [ ]:
client = openai.OpenAI()

def embed_text(texts):
    response = client.embeddings.create(
        input=texts,
        model="text-embedding-3-small"
    )
    return [np.array(e.embedding) for e in response.data]

embeddings = embed_text(chunks)

In [ ]:
dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [ ]:
def answer_query(query, k=3):
    # Embed query
    q_emb = embed_text([query])[0].reshape(1, -1)
    # Search FAISS
    D, I = index.search(q_emb, k)
    retrieved_chunks = [chunks[i] for i in I[0]]
    # Build context
    context = "\n\n".join(retrieved_chunks)
    # Ask LLM
    prompt = f"Answer the question based on the context below:\n\n{context}\n\nQuestion: {query}\nAnswer:"
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
print(answer_query("Summarize the main idea of chapter 1"))